Objetivo: Transformar os dados operacionais (OLTP) do sistema FatorV em um Data Warehouse (OLAP) para suportar decisões estratégicas sobre vendas e faturamento.

**Tecnologias utilizadas:**

- Linguagem: Python (Pandas para ETL).

- Origem: PostgreSQL (Sistema FatorV).

- Destino: Aiven Cloud (PostgreSQL/DW).

# Extração de Dados

Nesta etapa, conectamos ao banco de dados operacional para extrair as tabelas de Clientes, Produtos, Categorias, Fornecedores, Vendedores e Vendas.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

In [ ]:
uri = "INSIRA_SUA_URI_AQUI"

if uri.startswith("postgres://"):
  uri_final = uri.replace("postgres://", "postgresql://", 1)
else:
  uri_final = uri

In [ ]:
engine = create_engine(uri_final)

# "SELECT * FROM nome_da_tabela"
query_categoria = "SELECT * FROM categories"
query_customers = "SELECT * FROM customers"
query_products = "SELECT * FROM products"
query_sales = "SELECT * FROM sales"
query_sales_items = "SELECT * FROM sales_items"
query_sellers = "SELECT * FROM sellers"
query_suppliers = "SELECT * FROM suppliers"

df_categoria = pd.read_sql(query_categoria, engine)
df_customers = pd.read_sql(query_customers, engine)
df_products = pd.read_sql(query_products, engine)
df_sales = pd.read_sql(query_sales, engine)
df_sales_items = pd.read_sql(query_sales_items, engine)
df_sellers = pd.read_sql(query_sellers, engine)
df_suppliers = pd.read_sql(query_suppliers, engine)

# Transformação

In [ ]:
mapa_estados = {
    "Acre": ["acre", " ac ", "AC", "-ac", "/ac", "(ac)", "rio branco"],
    "Alagoas": ["alagoas", " al ", "AL", "-al", "/al", "(al)", "maceio"],
    "Amapá": ["amapá", "amapa", " ap ", "AP", "-ap", "/ap", "(ap)", "macapa"],
    "Amazonas": ["amazonas", " am ", "AM", "-am", "/am", "(am)", "manaus"],
    "Bahia": ["bahia", " ba ", "BA", "-ba", "/ba", "(ba)", "salvador", "camacari", "feira de santana"],
    "Ceará": ["ceará", "ceara", " ce ", "CE", "-ce", "/ce", "(ce)", "fortaleza"],
    "Distrito Federal": ["distrito federal", " df ", "DF", "-df", "/df", "(df)", "brasilia", "brasília"],
    "Espírito Santo": ["espírito santo", "espirito santo", " es ", "ES", "-es", "/es", "(es)", "vitoria", "vitória", "vila velha", "serra - es"],
    "Goiás": ["goiás", "goias", " go ", "GO", "-go", "/go", "(go)", "goiania", "aparecida de goiania"],
    "Maranhão": ["maranhão", "maranhao", " ma ", "MA", "-ma", "/ma", "(ma)", "sao luis"],
    "Mato Grosso": ["mato grosso", " mt ", "MT", "-mt", "/mt", "(mt)", "cuiaba"],
    "Mato Grosso do Sul": ["mato grosso do sul", " ms ", "MS", "-ms", "/ms", "(ms)", "campo grande"],
    "Minas Gerais": ["minas gerais", " mg ", "MG", "-mg", "/mg", "(mg)", "belo horizonte", "bh", "uberlandia", "contagem", "betim", "juiz de fora"],
    "Pará": ["pará", "para ", " pa ", "PA", "-pa", "/pa", "(pa)", "belem"],
    "Paraíba": ["paraíba", "paraiba", " pb ", "PB", "-pb", "/pb", "(pb)", "joao pessoa"],
    "Paraná": ["paraná", "parana", " pr ", "PR", "-pr", "/pr", "(pr)", "curitiba", "londrina", "maringa", "sao jose dos pinhais"],
    "Pernambuco": ["pernambuco", " pe ", "PE", "-pe", "/pe", "(pe)", "recife", "jaboatao", "olinda"],
    "Piauí": ["piauí", "piaui", " pi ", "PI", "-pi", "/pi", "(pi)", "teresina"],
    "Rio de Janeiro": ["rio de janeiro", " rj ", "RJ", "-rj", "/rj", "(rj)", "niteroi", "niterói", "duque de caxias", "nova iguacu"],
    "Rio Grande do Norte": ["rio grande do norte", " rn ", "RN", "-rn", "/rn", "(rn)", "natal"],
    "Rio Grande do Sul": ["rio grande do sul", " rs ", "RS", "-rs", "/rs", "(rs)", "porto alegre", "caxias do sul", "canoas"],
    "Rondônia": ["rondônia", "rondonia", " ro ", "RO", "-ro", "/ro", "(ro)", "porto velho"],
    "Roraima": ["roraima", " rr ", "RR", "-rr", "/rr", "(rr)", "boa vista"],
    "Santa Catarina": ["santa catarina", " sc ", "SC", "-sc", "/sc", "(sc)", "florianopolis", "blumenau", "joinville", "sao jose - sc"],
    "São Paulo": ["são paulo", "sao paulo", " sp ", "SP", " sp,", "-sp", " - sp", "/sp", "/ sp", "(sp)", "barueri", "alphaville", "osasco", "campinas", "guarulhos", "sbc", "bernardo", "santo andre", "sao caetano", "jundiai", "sorocaba", "ribeirao preto", "sao jose dos campos"],
    "Sergipe": ["sergipe", " se ", "SE", "-se", "/se", "(se)", "aracaju"],
    "Tocantins": ["tocantins", " to ", "TO", "-to", "/to", "(to)", "palmas"]
}

mapa_regioes = {
    # Norte
    "Acre": "Norte", "Amapá": "Norte", "Amazonas": "Norte",
    "Pará": "Norte", "Rondônia": "Norte", "Roraima": "Norte", "Tocantins": "Norte",

    # Nordeste
    "Alagoas": "Nordeste", "Bahia": "Nordeste", "Ceará": "Nordeste",
    "Maranhão": "Nordeste", "Paraíba": "Nordeste", "Pernambuco": "Nordeste",
    "Piauí": "Nordeste", "Rio Grande do Norte": "Nordeste", "Sergipe": "Nordeste",

    # Centro-Oeste
    "Distrito Federal": "Centro-Oeste", "Goiás": "Centro-Oeste",
    "Mato Grosso": "Centro-Oeste", "Mato Grosso do Sul": "Centro-Oeste",

    # Sudeste
    "Espírito Santo": "Sudeste", "Minas Gerais": "Sudeste",
    "Rio de Janeiro": "Sudeste", "São Paulo": "Sudeste",

    # Sul
    "Paraná": "Sul", "Rio Grande do Sul": "Sul", "Santa Catarina": "Sul"
}

def mapear_regiao(texto):
    if not texto: return "Não Informado"
    texto = str(texto).lower().strip()

    for estado, alias_list in mapa_estados.items():
        if any(alias.lower().strip() in texto for alias in alias_list):
            return mapa_regioes.get(estado)

    return "Não Identificado"

In [ ]:
def upper_case_df(df):
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.upper()
    return df

Deixando as colunas em maiusculo

In [ ]:
df_categoria = upper_case_df(df_categoria)
df_customers = upper_case_df(df_customers)
df_products = upper_case_df(df_products)
df_sellers = upper_case_df(df_sellers)
df_suppliers = upper_case_df(df_suppliers)

Criação dos estados

In [ ]:
df_customers['region'] = df_customers['state'].apply(mapear_regiao).str.upper()
df_suppliers['region'] = df_suppliers['state'].apply(mapear_regiao).str.upper()

Calculando subtotal e adicionando na tabela

In [ ]:
df_sales_items['subtotal'] = df_sales_items['quantity'] * df_sales_items['price']

In [ ]:
display(df_customers.head())
display(df_suppliers.head())
display(df_sales_items.head())

Dimensão Tempo no formato pedido

In [ ]:
datas = pd.to_datetime(df_sales['date']).unique()
df_tempo = pd.DataFrame({'data_completa': datas})

df_tempo['dia'] = df_tempo['data_completa'].dt.strftime('%Y%m%d').astype(int)
df_tempo['ano'] = df_tempo  ['data_completa'].dt.year
df_tempo['trimeste'] = df_tempo  ['data_completa'].dt.quarter
df_tempo['mes'] = df_tempo  ['data_completa'].dt.month
df_tempo

#MODELO DIMENSIONAL ESTRELA

Dimensão Produto

In [ ]:
df_dim_produto = df_products.merge(df_categoria, on='category_id').merge(df_suppliers, on='supplier_id')
df_dim_produto = df_dim_produto = df_products.merge(df_categoria, on='category_id') \
                            .merge(df_suppliers, on='supplier_id')

df_dim_produto

Tabela Fato

In [ ]:
df_sales = df_sales.rename(columns={'sale_id': 'sales_id'})
df_fato = df_sales.merge(df_sales_items, on='sales_id')

df_fato = df_fato.merge(df_sellers[['seller_id', 'tx_commission']], on='seller_id')
df_fato

Calcular o valor em dinheiro da comissão

In [ ]:
df_fato['valor_comissao'] = (df_fato['subtotal'] * df_fato['tx_commission']) / 100

df_fato['date_id'] = pd.to_datetime(df_fato['date']).dt.strftime('%Y%m%d').astype(int)
df_fato

In [ ]:
df_fato_vendas = df_fato[[
    'date_id',
    'sales_id',
    'customer_id',
    'product_id',
    'seller_id',
    'quantity',
    'price',
    'subtotal',
    'valor_comissao'
]]

display(df_fato_vendas.head())

Tratamento dos dados das tabelas novas

In [ ]:
df_dim_produto = df_products.merge(df_categoria, on='category_id') \
                            .merge(df_suppliers, on='supplier_id')

df_dim_produto = df_dim_produto[['product_id','product_name','category_name','supplier_id','state','region']]

In [ ]:
df_dim_cliente = df_customers[['customer_id','customer_name','state','region']].drop_duplicates()

In [ ]:
df_dim_fornecedor = df_suppliers[['supplier_id','supplier_name','state','region']].drop_duplicates()

In [ ]:
df_fato_vendas = df_fato_vendas.drop_duplicates()

# Carga de Dados (Load)

In [ ]:
from sqlalchemy import create_engine

uri_dw = "DATABASE_URL"

if uri_dw.startswith("postgres://"):
  uri_final_dw = uri_dw.replace("postgres://", "postgresql://", 1)
else:
  uri_final_dw = uri_dw

engine_dw = create_engine(uri_final_dw)

print("A iniciar a carga das Dimensões...")

df_tempo.to_sql('dim_tempo', engine_dw, if_exists='replace', index=False)
df_dim_produto.to_sql('dim_produto', engine_dw, if_exists='replace', index=False)
df_customers.to_sql('dim_cliente', engine_dw, if_exists='replace', index=False)
df_sellers.to_sql('dim_vendedor', engine_dw, if_exists='replace', index=False)
df_suppliers.to_sql('dim_fornecedor', engine_dw, if_exists='replace', index=False)
print("A iniciar a carga da Tabela Fato...")
df_fato_vendas.to_sql('fato_vendas', engine_dw, if_exists='replace', index=False)

print("\nCarga no Data Warehouse concluída com sucesso! ")

# Análise de Resultados

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

df_dw = df_fato_vendas.merge(df_dim_produto, on='product_id') \
                      .merge(df_sellers, on='seller_id')

sns.set_theme(style="whitegrid")
plt.figure(figsize=(20, 18))

plt.subplot(3, 2, 1)
top_produtos = df_dw.groupby('product_name')['quantity'].sum().nlargest(10).sort_values()
top_produtos.plot(kind='barh', color='teal')
plt.title('Top 10 Produtos Mais Vendidos (Qtd)', fontsize=14, fontweight='bold')
plt.xlabel('Quantidade Vendida')
plt.ylabel('Produto')


plt.subplot(3, 2, 2)
fat_total = df_dw['subtotal'].sum()
plt.text(0.5, 0.5, f'FATURAMENTO TOTAL\n\nR$ {fat_total:,.2f}',
         fontsize=24, ha='center', va='center', fontweight='bold', color='darkgreen',
         bbox=dict(facecolor='white', edgecolor='darkgreen', boxstyle='round,pad=1'))
plt.axis('off')


plt.subplot(3, 2, 3)
fat_categoria = df_dw.groupby('category_name')['subtotal'].sum().sort_values(ascending=False)
sns.barplot(x=fat_categoria.values, y=fat_categoria.index, hue=fat_categoria.index, palette='magma', legend=False)
plt.title('Faturamento por Categoria', fontsize=14, fontweight='bold')
plt.xlabel('Faturamento (R$)')
plt.ylabel('Categoria')


plt.subplot(3, 2, 4)
top_comissoes = df_dw.groupby('seller_name')['valor_comissao'].sum().nlargest(10).sort_values(ascending=False)
sns.barplot(x=top_comissoes.values, y=top_comissoes.index, hue=top_comissoes.index, palette='plasma', legend=False)
plt.title('Maiores Comissões por Vendedor', fontsize=14, fontweight='bold')
plt.xlabel('Comissão (R$)')
plt.ylabel('Vendedor')


plt.subplot(3, 2, 5)
forn_estado = df_suppliers['state'].value_counts()
sns.barplot(x=forn_estado.index, y=forn_estado.values, hue=forn_estado.index, palette='viridis', legend=False)
plt.title('Qtd. de Fornecedores por Estado', fontsize=14, fontweight='bold')
plt.ylabel('Quantidade')
plt.xlabel('Estado')


plt.subplot(3, 2, 6)
cli_estado = df_customers['state'].value_counts().nlargest(10)
sns.barplot(x=cli_estado.index, y=cli_estado.values, hue=cli_estado.index, palette='crest', legend=False)
plt.title('Top 10 Estados com Mais Clientes', fontsize=14, fontweight='bold')
plt.ylabel('Quantidade')
plt.xlabel('Estado')


plt.tight_layout()
plt.show()